In [4]:
from typing import List, Dict

In [5]:
class SlidingWindowHistoryManager:
    """
    Manages an active chat conversation history pool, automatically 
    evicting old turns to strictly enforce a target token capacity budget.
    """
    def __init__(self, max_token_budget: int):
        self.max_budget = max_token_budget
        self.history: List[Dict[str, str]] = []

    def _estimate_tokens(self, text: str) -> int:
        """
        A standard structural proxy for tokenization.
        Rough industry baseline: 1 token is approximately 4 characters.
        """
        return max(1, len(text) // 4)

    def add_message(self, role: str, content: str):
        """Appends a new turn to the history and automatically enforces the window boundary."""
        self.history.append({"role": role, "content": content})
        self._enforce_budget()

    def _enforce_budget(self):
        """Calculates total token volume and drops old turns if we exceed our limit."""
        while True:
            # Calculate the aggregate token weight currently sitting in our queue
            total_tokens = sum(self._estimate_tokens(msg["content"]) for msg in self.history)
            
            print(f"[WINDOW MANAGER] Current History Cost: {total_tokens}/{self.max_budget} tokens.")
            
            # If we are under budget, we are completely safe to break out
            if total_tokens <= self.max_budget:
                break
                
            # PITFALL WARNING: If we exceed budget, we must evict the OLDEST message 
            # sitting at index 0. This shifts our attention window forward in time.
            if self.history:
                removed = self.history.pop(0)
                print(f"  └─ [BUDGET EXCEEDED] Evicting oldest message turn: ({removed['role']}: '{removed['content'][:30]}...')")
            else:
                break

    def get_clean_payload(self) -> List[Dict[str, str]]:
        """Returns the fully managed, safely bounded history payload."""
        return self.history

In [6]:
# --- Execute Test Simulation ---
# Initialize a history manager with a very small token budget for visualization
manager = SlidingWindowHistoryManager(max_token_budget=50)

print("--- Step 1: Simulating early conversation turns ---")
manager.add_message("user", "Hello! I am setting up an automated embedded robotics rig.")
manager.add_message("assistant", "Excellent. Are you using an ESP32 micro-controller or an Arduino developer platform?")

print("\n--- Step 2: Injecting new long instructions (triggers eviction) ---")
# This massive payload will blow past the 50 token budget constraint
manager.add_message("user", "I chose the ESP32. I am writing a custom PID loop to balance a two-wheeled inverted pendulum chassis. I need to minimize oscillations across the derivative term to prevent motor driver overheating.")

print("\n--- Step 3: Inspecting final active context pool ---")
active_payload = manager.get_clean_payload()
print("\nFinal Array Delivered to LLM API:")
for index, msg in enumerate(active_payload):
    print(f"  [{index}] {msg['role'].upper()}: {msg['content']}")

--- Step 1: Simulating early conversation turns ---
[WINDOW MANAGER] Current History Cost: 14/50 tokens.
[WINDOW MANAGER] Current History Cost: 35/50 tokens.

--- Step 2: Injecting new long instructions (triggers eviction) ---
[WINDOW MANAGER] Current History Cost: 84/50 tokens.
  └─ [BUDGET EXCEEDED] Evicting oldest message turn: (user: 'Hello! I am setting up an auto...')
[WINDOW MANAGER] Current History Cost: 70/50 tokens.
  └─ [BUDGET EXCEEDED] Evicting oldest message turn: (assistant: 'Excellent. Are you using an ES...')
[WINDOW MANAGER] Current History Cost: 49/50 tokens.

--- Step 3: Inspecting final active context pool ---

Final Array Delivered to LLM API:
  [0] USER: I chose the ESP32. I am writing a custom PID loop to balance a two-wheeled inverted pendulum chassis. I need to minimize oscillations across the derivative term to prevent motor driver overheating.
